In [4]:
import pickle

file_path = "sp500_data.pkl"

# Open the file and then load
with open(file_path, 'rb') as f:  # 'rb' = read binary
    loaded_data = pickle.load(f)

# Now access the data
# If you saved it as a dictionary:
X_train = loaded_data['X_train']
y_train = loaded_data['y_train']
X_test = loaded_data['X_test']
y_test = loaded_data['y_test']

print(f"Loaded: X_train shape = {X_train.shape}")
print(f"Loaded: y_train shape = {y_train.shape}")

Loaded: X_train shape = torch.Size([1820, 60, 1])
Loaded: y_train shape = torch.Size([1820, 30, 1])


In [ ]:
import numpy as np
import pandas as pd
from darts import TimeSeries
from darts.models import AutoARIMA

# SIMPLEST: Use just one sequence for AutoARIMA
def create_single_timeseries_for_autoarima(X_train, y_train, sample_idx=0):
    """
    Create a single TimeSeries from one sample for AutoARIMA
    """
    # Get one sample
    X_sample = X_train[sample_idx].numpy().flatten()  # 60 values
    y_sample = y_train[sample_idx].numpy().flatten()  # 30 values
    
    # Combine
    sequence = np.concatenate([X_sample, y_sample])  # 90 values total
    
    # Create dates (use reasonable dates)
    dates = pd.date_range(
        start='2024-01-01',  # Arbitrary start date
        periods=len(sequence),
        freq='B'
    )
    
    # Create TimeSeries
    series = TimeSeries.from_times_and_values(
        times=dates,
        values=sequence,
        freq='B'
    )
    
    return series

# Create a single series
single_series = create_single_timeseries_for_autoarima(X_train, y_train, sample_idx=0)
print(f"Single series length: {len(single_series)}")

# Now use it with AutoARIMA
from darts.models import AutoARIMA

# Split into train/test (60 days train, 30 days test)
train = single_series[:60]  # First 60 values
test = single_series[60:]   # Last 30 values

# Train AutoARIMA
model = AutoARIMA(season_length=5)
model.fit(train)

# Predict
predictions = model.predict(len(test))

# Evaluate
from darts.metrics import rmse
print(f"RMSE: {rmse(test, predictions):.4f}")

Single series length: 90
RMSE: 0.0043


In [11]:
# Train AutoETS
from darts.models import AutoETS
model = AutoETS(season_length=5)
model.fit(train)

# Predict
predictions = model.predict(len(test))

# Evaluate
from darts.metrics import rmse
print(f"RMSE: {rmse(test, predictions):.4f}")

RMSE: 0.0042


In [13]:
# Train AutoTheta
from darts.models import AutoTheta
model = AutoETS(season_length=5)
model.fit(train)

# Predict
predictions = model.predict(len(test))

# Evaluate
from darts.metrics import rmse
print(f"RMSE: {rmse(test, predictions):.4f}")

RMSE: 0.0042
